In [ ]:
import numpy as np
import pandas as pd
import os

In [1]:

root_folder = '~/sfusd-local-data/zones/SFUSD/local_runs/comparisons/'
df = pd.read_csv(os.path.expanduser(f"{root_folder}/comparative_results.csv"))

NameError: name 'pd' is not defined

In [10]:
df['objective_value'] = df['objective_value'].replace(-1, np.nan)
df = df[df['level']== 'BlockGroup']
df

,time_limit,seed,centroids_type,level,frl_dev,racial_dev,optimizer,status,wall_time,objective_value
0,300,42,13-zone-6,BlockGroup,0.2,0.3,cp_int,FEASIBLE,300.823993,259.0
1,60,2025,8-zone-25,BlockGroup,0.2,0.3,cp_bool,FEASIBLE,60.184356,210.0
2,600,1014,6-zone-2,BlockGroup,0.4,0.3,cp_bool,FEASIBLE,600.206913,126.0
3,60,2025,13-zone-6,BlockGroup,0.2,0.3,mip,TIME_LIMIT,60.002120,NaN
4,300,42,6-zone-2,BlockGroup,0.4,0.3,mip,TIME_LIMIT,300.020669,111.0
...,...,...,...,...,...,...,...,...,...,...
268,300,42,8-zone-25,BlockGroup,0.4,0.3,cp_int,FEASIBLE,300.253259,154.0
269,300,2025,13-zone-6,BlockGroup,0.4,0.3,cp_int,FEASIBLE,300.298818,236.0
270,60,42,10-zone-3,BlockGroup,0.2,0.3,cp_int,FEASIBLE,60.313164,209.0
271,600,1014,10-zone-3,BlockGroup,0.4,0.3,cp_bool,FEASIBLE,600.383257,210.0


In [11]:
grouped_dfs = df.groupby(['seed', 'centroids_type', 'level', 'time_limit','frl_dev', 'racial_dev'], as_index=False)

In [12]:
for name, group in grouped_dfs:
    print(group)
    print("\n")

     time_limit  seed centroids_type       level  frl_dev  racial_dev  \
171          60    42      10-zone-3  BlockGroup      0.2         0.3   
261          60    42      10-zone-3  BlockGroup      0.2         0.3   
270          60    42      10-zone-3  BlockGroup      0.2         0.3   

    optimizer      status  wall_time  objective_value  
171   cp_bool    FEASIBLE  60.485561            235.0  
261       mip  TIME_LIMIT  60.010583              NaN  
270    cp_int    FEASIBLE  60.313164            209.0  


     time_limit  seed centroids_type       level  frl_dev  racial_dev  \
92           60    42      10-zone-3  BlockGroup      0.4         0.3   
161          60    42      10-zone-3  BlockGroup      0.4         0.3   
182          60    42      10-zone-3  BlockGroup      0.4         0.3   

    optimizer      status  wall_time  objective_value  
92     cp_int    FEASIBLE  60.242409            215.0  
161       mip  TIME_LIMIT  60.014519              NaN  
182   cp_bool    FEA

In [13]:
# find average percent difference in objective value between cp_int and cp_bool for each group
results = []
for name, group in grouped_dfs:
    cp_int = group[group['optimizer'] == 'cp_int']['objective_value'].values
    cp_bool = group[group['optimizer'] == 'cp_bool']['objective_value'].values
    mip = group[group['optimizer'] == 'mip']['objective_value'].values
    bool_gap = np.nan
    mip_gap = np.nan
    # bool gap is the percent difference between cp_bool and cp_int
    # mip gap is the percent difference between mip and cp_int
    if len(cp_int) > 0 and len(cp_bool) > 0:
        bool_gap = (cp_bool[0] - cp_int[0]) / cp_int[0] * 100
    if len(cp_int) > 0 and len(mip) > 0:
        mip_gap = (mip[0] - cp_int[0]) / cp_int[0] * 100
    results.append(list(name) + [cp_int[0] if len(cp_int) > 0 else np.nan,
                                  cp_bool[0] if len(cp_bool) > 0 else np.nan,
                                  mip[0] if len(mip) > 0 else np.nan,
                                  bool_gap, mip_gap])

results_df = pd.DataFrame(results, columns=['seed', 'centroids_type', 'level', 'time_limit','frl_dev', 'racial_dev', 'cp_int_obj', 'cp_bool_obj', 'mip_obj', 'bool_gap', 'mip_gap'])


In [14]:
results_df

,seed,centroids_type,level,time_limit,frl_dev,racial_dev,cp_int_obj,cp_bool_obj,mip_obj,bool_gap,mip_gap
0,42,10-zone-3,BlockGroup,60,0.2,0.3,209.0,235.0,NaN,12.440191,NaN
1,42,10-zone-3,BlockGroup,60,0.4,0.3,215.0,230.0,NaN,6.976744,NaN
2,42,10-zone-3,BlockGroup,300,0.2,0.3,214.0,229.0,NaN,7.009346,NaN
3,42,10-zone-3,BlockGroup,300,0.4,0.3,190.0,195.0,NaN,2.631579,NaN
4,42,10-zone-3,BlockGroup,600,0.2,0.3,213.0,206.0,NaN,-3.286385,NaN
...,...,...,...,...,...,...,...,...,...,...,...
85,2025,8-zone-25,BlockGroup,60,0.4,0.3,174.0,214.0,NaN,22.988506,NaN
86,2025,8-zone-25,BlockGroup,300,0.2,0.3,162.0,169.0,NaN,4.320988,NaN
87,2025,8-zone-25,BlockGroup,300,0.4,0.3,167.0,177.0,190.0,5.988024,13.772455
88,2025,8-zone-25,BlockGroup,600,0.2,0.3,178.0,175.0,NaN,-1.685393,NaN
